# CelebA Face Attribute Classification

End-to-end pipeline for classifying facial attributes (Smiling) on the CelebA dataset.
Integrates MySQL storage with deep convolutional neural networks and gradient boosted trees (XGBoost), using 5-fold cross-validation and test-time augmentation (TTA).



In [ ]:
import os
import sys

os.environ.setdefault('KERAS_HOME', '.')

import numpy as np
import pandas as pd
import pymysql
from PIL import Image
from scipy.optimize import minimize
from sklearn.metrics import accuracy_score, f1_score, log_loss, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, train_test_split

try:
    from xgboost import XGBClassifier
except ImportError:
    XGBClassifier = None

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers



: 

## 1. Load Data
Fetches metadata and attribute annotations from MySQL. If the database is offline, reads directly from the local CSV fallback.



In [ ]:
def load_data(sample_size=20000, target_attr='Smiling', include_metadata=False):
    """
    Unified data loader querying the master view joining images_metadata, celebrities,
    celeba_identities, and celeba_attributes from MySQL.
    """
    try:
        conn = pymysql.connect(
            host='127.0.0.1', port=3306, user='root', password='',
            database='celeba_db', connect_timeout=3
        )
        query = f"""
            SELECT
                file_path,
                `{target_attr}` AS label,
                celebrity_name,
                identity_cluster,
                gender,
                age_group
            FROM celeba_master_view
            WHERE `{target_attr}` IS NOT NULL
            LIMIT {sample_size};
        """
        df = pd.read_sql(query, conn)
        conn.close()
        print(f"Loaded {len(df)} records from MySQL celeba_master_view (Joined 4 Tables).")
    except Exception as e:
        print(f"MySQL offline ({e}). Using list_attr_celeba.csv fallback.")
        df = pd.read_csv('data/metadata/list_attr_celeba.csv', nrows=sample_size)
        df['file_path'] = df['image_id'].apply(
            lambda name: os.path.join('img_align_celeba', name.replace('.jpg', '.png'))
        )
        df['label'] = df[target_attr]
        df['celebrity_name'] = 'Unknown'
        df['identity_cluster'] = 0
        df['gender'] = df['Male'].apply(lambda x: 'Male' if x == 1 else 'Female')
        df['age_group'] = df['Young'].apply(lambda x: 'Young' if x == 1 else 'Senior')

    # Convert binary attributes (-1, 1) to (0, 1)
    df['label'] = (df['label'] == 1).astype(int)
    if include_metadata:
        return df[['file_path', 'label', 'celebrity_name', 'identity_cluster', 'gender', 'age_group']].copy()
    return df[['file_path', 'label']].copy()

df = load_data(sample_size=20000, target_attr='Smiling', include_metadata=True)
print(f"Dataset shape: {df.shape}")
print(df[['celebrity_name', 'gender', 'age_group', 'label']].head())


## 2. Preprocess Images
Resizes images to 64x64, scales pixels, and splits into train (70%), validation (15%), and test (15%) partitions.
Standardization parameters are computed only on the training set to prevent leakage.



In [ ]:
IMG_SIZE = 64
CHANNELS = 3

def load_and_preprocess(data_df, scaling='standardize', test_size=0.15, val_size=0.15, random_state=42):
    images, labels = [], []

    for _, row in data_df.iterrows():
        path = row['file_path']
        if os.path.exists(path):
            try:
                img = Image.open(path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
                images.append(np.asarray(img, dtype=np.float32) / 255.0)
                labels.append(row['label'])
            except Exception:
                continue

    X = np.array(images, dtype=np.float32)
    y = np.array(labels, dtype=np.float32)

    holdout = test_size + val_size
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=holdout, random_state=random_state, stratify=y
    )

    val_ratio = val_size / holdout
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=(1.0 - val_ratio), random_state=random_state, stratify=y_temp
    )

    if scaling == 'standardize':
        mean = X_train.mean(axis=(0, 1, 2), keepdims=True)
        std = X_train.std(axis=(0, 1, 2), keepdims=True) + 1e-7
        X_train = (X_train - mean) / std
        X_val = (X_val - mean) / std
        X_test = (X_test - mean) / std

    print(f"Dataset split: train={len(X_train)}, val={len(X_val)}, test={len(X_test)}")
    return (X_train, y_train), (X_val, y_val), (X_test, y_test)

(X_train, y_train), (X_val, y_val), (X_test, y_test) = load_and_preprocess(df, scaling='standardize')



## 3. Model Architectures
- Real-time augmentation: random horizontal flip, slight rotation and zoom
- Residual CNN: 4 convolution blocks with skip connections and batch normalization
- XGBoost: gradient boosted decision trees on flattened pixel arrays



In [ ]:
augmenter = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.04),
    layers.RandomZoom(0.05),
], name="augmenter")

def build_residual_cnn():
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, CHANNELS))
    x = augmenter(inputs)

    # block 1 (32 filters)
    c1 = layers.Conv2D(32, 3, padding='same', activation='relu')(x)
    c1 = layers.BatchNormalization()(c1)
    c1 = layers.Conv2D(32, 3, padding='same', activation='relu')(c1)
    c1 = layers.BatchNormalization()(c1)
    skip1 = layers.Conv2D(32, 1, padding='same')(x)
    x = layers.MaxPooling2D(2)(layers.add([c1, skip1]))

    # block 2 (64 filters)
    c2 = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    c2 = layers.BatchNormalization()(c2)
    c2 = layers.Conv2D(64, 3, padding='same', activation='relu')(c2)
    c2 = layers.BatchNormalization()(c2)
    skip2 = layers.Conv2D(64, 1, padding='same')(x)
    x = layers.MaxPooling2D(2)(layers.add([c2, skip2]))

    # block 3 (128 filters)
    c3 = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    c3 = layers.BatchNormalization()(c3)
    c3 = layers.Conv2D(128, 3, padding='same', activation='relu')(c3)
    c3 = layers.BatchNormalization()(c3)
    skip3 = layers.Conv2D(128, 1, padding='same')(x)
    x = layers.MaxPooling2D(2)(layers.add([c3, skip3]))

    # block 4 (256 filters) + pooling
    x = layers.Conv2D(256, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(64, activation='relu')(x)
    out = layers.Dense(1, activation='sigmoid')(x)

    model = keras.Model(inputs=inputs, outputs=out, name="residual_cnn")
    model.compile(optimizer=keras.optimizers.Adam(3e-4), loss='binary_crossentropy', metrics=['accuracy'])
    return model

def build_xgboost():
    if XGBClassifier is None:
        raise ImportError("xgboost is not installed")
    return XGBClassifier(
        n_estimators=120,
        max_depth=5,
        learning_rate=0.08,
        reg_lambda=1.0,
        n_jobs=-1,
        random_state=42
    )



## 4. 5-Fold Cross-Validation, Blending, and TTA
Runs 5-fold cross-validation on XGBoost and the residual CNN, blends out-of-fold probabilities via SLSQP optimization, and evaluates test-time augmentation (horizontal flips).



In [ ]:
X_full = np.concatenate([X_train, X_val], axis=0)
y_full = np.concatenate([y_train, y_val], axis=0)

X_full_flat = X_full.reshape(len(X_full), -1)
X_test_flat = X_test.reshape(len(X_test), -1)

# baseline single-fold xgboost
xgb_baseline = build_xgboost()
xgb_baseline.fit(X_full_flat[:1400], y_full[:1400])
p_test_xgb_single = xgb_baseline.predict_proba(X_test_flat)[:, 1]

# 5-fold cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_xgb = np.zeros(len(X_full))
oof_cnn = np.zeros(len(X_full))
xgb_test_preds = []
cnn_test_preds = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_full, y_full), 1):
    print(f"Fold {fold}/5...")

    # xgb
    xgb = build_xgboost()
    xgb.fit(X_full_flat[tr_idx], y_full[tr_idx])
    oof_xgb[val_idx] = xgb.predict_proba(X_full_flat[val_idx])[:, 1]
    xgb_test_preds.append(xgb.predict_proba(X_test_flat)[:, 1])

    # cnn
    cnn = build_residual_cnn()
    cnn.fit(
        X_full[tr_idx], y_full[tr_idx],
        validation_data=(X_full[val_idx], y_full[val_idx]),
        epochs=5, batch_size=32, verbose=0
    )
    oof_cnn[val_idx] = cnn.predict(X_full[val_idx], verbose=0).flatten()
    cnn_test_preds.append(cnn.predict(X_test, verbose=0).flatten())

p_test_xgb_bagged = np.mean(xgb_test_preds, axis=0)
p_test_cnn_bagged = np.mean(cnn_test_preds, axis=0)

# optimize ensemble weights on validation log-loss
val_preds = np.column_stack([oof_xgb, oof_cnn])
test_preds = np.column_stack([p_test_xgb_bagged, p_test_cnn_bagged])

def loss_func(w):
    w = w / np.sum(w)
    blend = np.clip(val_preds @ w, 1e-7, 1 - 1e-7)
    return log_loss(y_full, blend)

res = minimize(loss_func, [0.8, 0.2], method='SLSQP', bounds=[(0, 1), (0, 1)], constraints={'type': 'eq', 'fun': lambda w: 1 - np.sum(w)})
weights = res.x / np.sum(res.x)
print(f"Optimal weights: XGBoost={weights[0]:.4f}, Residual CNN={weights[1]:.4f}")

p_test_ensemble = test_preds @ weights

# test-time augmentation (horizontal flip)
X_test_flipped = np.flip(X_test, axis=2)
X_test_flipped_flat = X_test_flipped.reshape(len(X_test_flipped), -1)

p_test_xgb_flip = np.mean([m.predict_proba(X_test_flipped_flat)[:, 1] for m in [xgb]], axis=0)
p_test_cnn_flip = cnn.predict(X_test_flipped, verbose=0).flatten()
p_test_flip_ensemble = weights[0] * p_test_xgb_flip + weights[1] * p_test_cnn_flip

p_test_tta = 0.5 * (p_test_ensemble + p_test_flip_ensemble)

def get_metrics(name, y_true, probs):
    preds = (probs > 0.5).astype(int)
    loss = log_loss(y_true, np.clip(probs, 1e-7, 1 - 1e-7))
    return {
        'Model Configuration': name,
        'Test Loss': round(loss, 4),
        'Accuracy': f"{accuracy_score(y_true, preds)*100:.2f}%",
        'Precision': round(precision_score(y_true, preds, zero_division=0), 4),
        'Recall': round(recall_score(y_true, preds, zero_division=0), 4),
        'F1-Score': round(f1_score(y_true, preds, zero_division=0), 4)
    }

results = [
    get_metrics('Single-Fold XGBoost Baseline', y_test, p_test_xgb_single),
    get_metrics('5-Fold Bagged XGBoost', y_test, p_test_xgb_bagged),
    get_metrics('5-Fold Bagged Augmented CNN', y_test, p_test_cnn_bagged),
    get_metrics('5-Fold Ensemble (XGBoost + Augmented CNN)', y_test, p_test_ensemble),
    get_metrics('5-Fold Ensemble + TTA (Test-Time Augmentation)', y_test, p_test_tta)
]

results_df = pd.DataFrame(results)
print("\nBenchmark Results:")
print(results_df.to_string(index=False))
results_df.to_csv('reports/model_benchmark_results.csv', index=False)



## 5. Visualizations & Performance Analytics
Visualizes model performance metrics, demographic attribute distributions, and the evaluation confusion matrix with ROC curves using Matplotlib.


In [ ]:
# Cell 6: Matplotlib Performance & Attribute Visualizations
import os

import matplotlib.pyplot as plt

# Ensure output directory exists
figures_dir = 'reports/figures'
os.makedirs(figures_dir, exist_ok=True)

# 1. Plot Benchmark Comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), dpi=150)

model_labels = ['Single XGB', '5-Fold XGB', 'Aug CNN', 'Ensemble', 'Ensemble+TTA']
acc_vals = [float(str(x).replace('%', '')) for x in results_df['Accuracy']]
f1_vals = [float(x) * 100 for x in results_df['F1-Score']]
losses = [float(x) for x in results_df['Test Loss']]

x = np.arange(len(model_labels))
width = 0.35
ax1.bar(x - width/2, acc_vals, width, label='Accuracy (%)', color='#2563eb')
ax1.bar(x + width/2, f1_vals, width, label='F1-Score (%)', color='#10b981')
ax1.set_ylabel('Score (%)', fontweight='bold')
ax1.set_title('Model Performance Comparison', fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(model_labels)
ax1.set_ylim(0, 105)
ax1.legend()
ax1.grid(axis='y', linestyle='--', alpha=0.3)

# 2. Plot Test Log Loss
ax2.bar(model_labels, losses, color=['#3b82f6', '#1d4ed8', '#ef4444', '#10b981', '#059669'], width=0.5)
ax2.set_ylabel('Binary Cross-Entropy Loss', fontweight='bold')
ax2.set_title('Test Loss across Models (Lower is Better)', fontweight='bold')
ax2.grid(axis='y', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(figures_dir, 'model_benchmark_comparison.png'), bbox_inches='tight')
plt.show()
print(f"Charts successfully rendered and saved to {figures_dir}/")
